In [ ]:
# Auto-generated by kaggle_tunnel_app.py
# Run this notebook cell on the remote machine that should connect back to the PC.

import asyncio
import base64
import importlib
import json
import socket
import subprocess
import sys
from dataclasses import dataclass

def ensure_package(module_name: str, package_name=None):
    try:
        return importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name or module_name])
        return importlib.import_module(module_name)


nest_asyncio = ensure_package("nest_asyncio")
websockets = ensure_package("websockets")

nest_asyncio.apply()

CONTROL_URL = 'wss://inherited-buddy-conclude-timing.trycloudflare.com/ws?token=mFhBAfOgZ1I90vlIvBIGUlu28bh8cfnU'
CONTROL_TOKEN = 'mFhBAfOgZ1I90vlIvBIGUlu28bh8cfnU'


def _b64encode(data: bytes) -> str:
    return base64.b64encode(data).decode("ascii")


def _b64decode(text: str) -> bytes:
    return base64.b64decode(text.encode("ascii"))


@dataclass
class TcpHandle:
    reader: asyncio.StreamReader
    writer: asyncio.StreamWriter
    pump_task: asyncio.Task


class RemoteAgent:
    def __init__(self, control_url: str, control_token: str):
        self.control_url = control_url
        self.control_token = control_token
        self.websocket = None
        self.tcp_handles = {}
        self.ssh_processes = {}

    async def send(self, payload):
        if self.websocket is not None:
            await self.websocket.send(json.dumps(payload))

    async def handle_run_command(self, payload):
        request_id = payload["request_id"]
        process = await asyncio.create_subprocess_shell(
            payload["command"],
            stdout=asyncio.subprocess.PIPE,
            stderr=asyncio.subprocess.PIPE,
        )
        stdout, stderr = await process.communicate()
        await self.send(
            {
                "type": "command_result",
                "request_id": request_id,
                "returncode": process.returncode,
                "stdout": stdout.decode("utf-8", errors="replace"),
                "stderr": stderr.decode("utf-8", errors="replace"),
            }
        )

    async def _tcp_pump(self, connection_id: str, reader: asyncio.StreamReader):
        try:
            while True:
                chunk = await reader.read(65536)
                if not chunk:
                    break
                await self.send(
                    {
                        "type": "tcp_data",
                        "connection_id": connection_id,
                        "data": _b64encode(chunk),
                    }
                )
        finally:
            await self.close_tcp(connection_id)

    async def handle_tcp_open(self, payload):
        connection_id = payload["connection_id"]
        try:
            reader, writer = await asyncio.open_connection(payload["host"], int(payload["port"]))
            self.tcp_handles[connection_id] = TcpHandle(
                reader=reader,
                writer=writer,
                pump_task=asyncio.create_task(self._tcp_pump(connection_id, reader)),
            )
            await self.send({"type": "tcp_opened", "connection_id": connection_id})
        except Exception as exc:
            await self.send(
                {
                    "type": "tcp_closed",
                    "connection_id": connection_id,
                    "error": str(exc),
                }
            )

    async def handle_tcp_data(self, payload):
        handle = self.tcp_handles.get(payload["connection_id"])
        if handle is None:
            return
        handle.writer.write(_b64decode(payload["data"]))
        await handle.writer.drain()

    async def close_tcp(self, connection_id: str):
        handle = self.tcp_handles.pop(connection_id, None)
        if handle is None:
            return
        handle.writer.close()
        try:
            await handle.writer.wait_closed()
        except Exception:
            pass
        if not handle.pump_task.done():
            handle.pump_task.cancel()
        await self.send({"type": "tcp_closed", "connection_id": connection_id})

    async def handle_start_ssh_tunnel(self, payload):
        request_id = payload["request_id"]
        command = [
            "ssh",
            "-N",
            "-p",
            str(int(payload.get("ssh_port", 22))),
            "-o",
            "ExitOnForwardFailure=yes",
            "-o",
            "ServerAliveInterval=30",
        ]
        for spec in payload.get("forwards", []):
            command.extend(["-L", spec])
        command.extend(payload.get("extra_args", []))
        command.append(payload["ssh_target"])
        try:
            process = await asyncio.create_subprocess_exec(
                *command,
                stdout=asyncio.subprocess.PIPE,
                stderr=asyncio.subprocess.PIPE,
            )
            self.ssh_processes[request_id] = process
            await self.send(
                {
                    "type": "ssh_status",
                    "request_id": request_id,
                    "state": "started",
                    "pid": process.pid,
                    "command": command,
                }
            )
        except Exception as exc:
            await self.send(
                {
                    "type": "ssh_status",
                    "request_id": request_id,
                    "state": "failed",
                    "error": str(exc),
                }
            )

    async def handle_stop_ssh_tunnel(self, payload):
        request_id = payload["request_id"]
        process = self.ssh_processes.pop(request_id, None)
        if process is None:
            await self.send({"type": "ssh_status", "request_id": request_id, "state": "missing"})
            return
        process.terminate()
        await process.wait()
        await self.send(
            {
                "type": "ssh_status",
                "request_id": request_id,
                "state": "stopped",
                "returncode": process.returncode,
            }
        )

    async def handle_message(self, payload):
        message_type = payload.get("type")
        if message_type == "run_command":
            await self.handle_run_command(payload)
        elif message_type == "tcp_open":
            await self.handle_tcp_open(payload)
        elif message_type == "tcp_data":
            await self.handle_tcp_data(payload)
        elif message_type == "tcp_close":
            await self.close_tcp(payload["connection_id"])
        elif message_type == "start_ssh_tunnel":
            await self.handle_start_ssh_tunnel(payload)
        elif message_type == "stop_ssh_tunnel":
            await self.handle_stop_ssh_tunnel(payload)

    async def run(self):
        headers = [("x-kaggle-tunnel-token", self.control_token)]
        while True:
            try:
                async with websockets.connect(
                    self.control_url,
                    additional_headers=headers,
                    ping_interval=20,
                    ping_timeout=20,
                    max_size=None,
                ) as websocket:
                    self.websocket = websocket
                    await self.send(
                        {
                            "type": "hello",
                            "hostname": socket.gethostname(),
                            "python": sys.version,
                            "platform": sys.platform,
                        }
                    )
                    async for raw in websocket:
                        await self.handle_message(json.loads(raw))
            except Exception as exc:
                print(f"remote agent reconnecting after error: {exc}")
                await asyncio.sleep(2)
            finally:
                self.websocket = None


agent = RemoteAgent(CONTROL_URL, CONTROL_TOKEN)
asyncio.get_event_loop().create_task(agent.run())
print("Remote notebook agent started. Leave this cell running.")
print("If the cell finishes, rerun it to restart the background tasks.")
